In [0]:
files = {
    "employee" : "/Volumes/workspace/default/assignment/employees_extended.json",
    "Transaction" : "/Volumes/workspace/default/assignment/transactions_1200 (1).json"
}
df = {}

for name, path in files.items():
    df[name]=spark.read.json(
        path
    )

In [0]:
df["employee"].select("name").display()

name
Ravi K
Anu S
Anu V
Kiran N
Divya B
Vikram V
Ravi J
Karthik M
Ravi C
Manoj V


### Q13: Data Cleaning 

> **Remove duplicate employees  **

In [0]:
df["employee"].printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- join_date: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- level: string (nullable = true)
 |    |-- region: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- projects: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- hours: long (nullable = true)
 |    |    |-- project_name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
df["employee"] = df["employee"].dropDuplicates(["emp_id"])
df["employee"].show()

+---------------+----+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|        address| age|department|emp_id|is_active| join_date|            metadata|     name|            projects|   salary|              skills|
+---------------+----+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|    {Delhi, DL}|  41|Operations|    28|    false|2022-08-31|    {NULL, IN, NULL}|  Aisha M|[{483, Delta}, {3...| 51744.11|             [Scala]|
|   {Mumbai, MH}|  54|   Support|    29|    false|2024-02-15|{NULL, NULL, manual}|Karthik J|[{268, Delta}, {2...|136455.19|        [AWS, Scala]|
|{Bangalore, KA}|NULL|        IT|    30|    false|2022-10-22|       {L2, US, api}|   Neha V|                  []|138366.11|        [AWS, Spark]|
|    {Kochi, KL}|  24|   Finance|    33|    false|2025-01-13|    {NULL, EU, NULL}|  Rahul C|                  []| 74581.81|       

> Handle null values in salary and age  

In [0]:
from pyspark.sql.functions import col

df["employee"].filter(
    col("salary").isNull()
).select("emp_id","name").show()


+------+---------+
|emp_id|     name|
+------+---------+
|     2|    Anu S|
|     4|  Kiran N|
|    11|   Ravi H|
|    12|   John J|
|    16|    Anu S|
|    17|  Priya V|
|    21|  Priya L|
|    34|  Divya R|
|    39|  Priya L|
|    41|   Ravi M|
|    43|  Aisha D|
|    52|  Rahul A|
|    61|  Divya K|
|    62|    Anu B|
|    64|   Ravi B|
|    66|  Manoj T|
|    70|   John H|
|    80|  Manoj V|
|    81|  Priya H|
|    82|Karthik P|
+------+---------+
only showing top 20 rows


In [0]:
df["employee"]= df["employee"].fillna(250000,subset=["salary"]).show()

+---------------+----+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|        address| age|department|emp_id|is_active| join_date|            metadata|     name|            projects|   salary|              skills|
+---------------+----+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|{Hyderabad, TS}|  26|Operations|     1|    false|2024-02-12|  {L3, NULL, manual}|   Ravi K|[{35, Alpha}, {13...| 59378.66|               [GCP]|
|{Bangalore, KA}|  30|     Sales|     2|     true|2021-10-25|  {L2, NULL, ingest}|    Anu S|                  []| 250000.0|[Spark, Scala, Ex...|
|{Hyderabad, TS}|  55|        IT|     3|     true|2025-01-18|   {NULL, NULL, api}|    Anu V|                  []|146639.47|   [GCP, Scala, SQL]|
|    {Delhi, DL}|  42|Operations|     4|     true|2020-12-29|     {L3, NULL, api}|  Kiran N|    [{393, Epsilon}]| 250000.0|       

In [0]:
median_age = df["employee"].approxQuantile(
    "age", [0.5], 0
)[0]

In [0]:
df["employee"] = df["employee"].fillna(median_age,"age")
df["employee"].show()

+---------------+---+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|        address|age|department|emp_id|is_active| join_date|            metadata|     name|            projects|   salary|              skills|
+---------------+---+----------+------+---------+----------+--------------------+---------+--------------------+---------+--------------------+
|{Hyderabad, TS}| 26|Operations|     1|    false|2024-02-12|  {L3, NULL, manual}|   Ravi K|[{35, Alpha}, {13...| 59378.66|               [GCP]|
|{Bangalore, KA}| 30|     Sales|     2|     true|2021-10-25|  {L2, NULL, ingest}|    Anu S|                  []|     NULL|[Spark, Scala, Ex...|
|{Hyderabad, TS}| 55|        IT|     3|     true|2025-01-18|   {NULL, NULL, api}|    Anu V|                  []|146639.47|   [GCP, Scala, SQL]|
|    {Delhi, DL}| 42|Operations|     4|     true|2020-12-29|     {L3, NULL, api}|  Kiran N|    [{393, Epsilon}]|     NULL|              

### Q14: withColumn Usage

In [0]:
from pyspark.sql.functions import expr
df["employee"] = df["employee"].withColumn(
    "join_date",
    expr("try_cast(join_date as date)")
)

In [0]:
df["employee"].printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- join_date: date (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- level: string (nullable = true)
 |    |-- region: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- projects: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- hours: long (nullable = true)
 |    |    |-- project_name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
from pyspark.sql.functions import current_date, datediff, col, round

df["employee"].withColumn(
    "Exprience_year",
    round(datediff(current_date(),col("join_date"))/365,1)
).display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills,Exprience_year
"List(Hyderabad, TS)",26,Operations,1,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP),2.2
"List(Bangalore, KA)",30,Sales,2,true,2021-10-25,"List(L2, null, ingest)",Anu S,List(),null,"List(Spark, Scala, Excel)",4.5
"List(Hyderabad, TS)",55,IT,3,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)",1.3
"List(Delhi, DL)",42,Operations,4,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP),5.4
"List(Hyderabad, TS)",22,Support,5,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)",3.9
"List(Hyderabad, TS)",57,Sales,6,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)",5.9
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau),8.2
"List(Kochi, KL)",39,Marketing,8,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)",2.7
"List(Chennai, TN)",39,Sales,9,false,2020-09-13,"List(null, EU, api)",Ravi C,"List(List(55, Omega), List(292, Omega), List(84, Omega))",134840.23,List(SQL),5.7
"List(Bangalore, KA)",39,Support,10,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)",0.8


### Q15: Complex Column Transformation 

In [0]:
from pyspark.sql.functions import expr

df["employee"].withColumn(
    "Seniority",
    expr("case when age > 35 and salary >70000 then 'Senior' end")
).display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills,Seniority
"List(Hyderabad, TS)",26,Operations,1,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP),null
"List(Bangalore, KA)",30,Sales,2,true,2021-10-25,"List(L2, null, ingest)",Anu S,List(),null,"List(Spark, Scala, Excel)",null
"List(Hyderabad, TS)",55,IT,3,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)",Senior
"List(Delhi, DL)",42,Operations,4,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP),null
"List(Hyderabad, TS)",22,Support,5,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)",null
"List(Hyderabad, TS)",57,Sales,6,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)",Senior
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau),Senior
"List(Kochi, KL)",39,Marketing,8,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)",Senior
"List(Chennai, TN)",39,Sales,9,false,2020-09-13,"List(null, EU, api)",Ravi C,"List(List(55, Omega), List(292, Omega), List(84, Omega))",134840.23,List(SQL),Senior
"List(Bangalore, KA)",39,Support,10,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)",Senior
